# AI Data Quality Reporter (GlobalMart)

## 1) Install OpenAI Python SDK (Once per cluster) and Imports

In [0]:
%pip install openai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json,re
from datetime import datetime, timezone
from openai import OpenAI
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

## 2) Configuration, and UC table

In [0]:

MODEL_NAME = "databricks-gpt-oss-20b"

# Same catalog/schema pattern as Code/Silver.py and Code/Gold.py
SILVER_CAT = "globalmart.silver"
GOLD_CAT = "globalmart.gold"
GOLD_TABLE = f"{GOLD_CAT}.dq_audit_report"

# Quarantine tables — same entity ↔ quarantine pairs as quality_run_summary in Code/Silver.py
QUARANTINE_TABLE_PAIRS = [
    ("customers", "quarantine_customers"),
    ("orders", "quarantine_orders"),
    ("transactions", "quarantine_transactions"),
    ("returns", "quarantine_returns"),
    ("products", "quarantine_products"),
    ("vendors", "quarantine_vendors"),
]
QUARANTINE_SOURCES = [(f"{SILVER_CAT}.{q}", e) for e, q in QUARANTINE_TABLE_PAIRS]

# Maps Silver `failure_reasons` token → primary field (keep in sync with Code/Silver.py concat_ws rules)
ISSUE_TO_FIELD = {
    # customers
    "null_customer_id": "customer_id",
    "Not_valid_segment_or_null": "segment",
    "non_us_country": "country",
    # orders
    "null_order_id": "order_id",
    "unparseable_order_purchase_date": "order_purchase_date",
    "null_ship_mode": "ship_mode",
    "null_vendor_id": "vendor_id",
    "invalid_vendor_id": "vendor_id",
    # transactions
    "null_product_id": "product_id",
    "null_quantity": "quantity",
    "unparseable_sales": "sales_amount",
    "null_payment_type": "payment_type",
    "negative_sales": "sales_amount",
    # returns
    "null_refund_amount_or_not_valid": "refund_amount",
    "unparseable_return_date": "return_date",
    "invalid_return_status": "return_status",
    "null_return_reason_or_not_valid": "return_reason",
    # products
    "empty_product_name": "product_name",
    # vendors
    "null_vendor_name": "vendor_name",
}

spark.sql(
    f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
  entity_name STRING COMMENT 'Silver entity: customers, orders, transactions, returns, products, vendors',
  field_affected STRING COMMENT 'Primary column or measure implicated by the DQ rule',
  issue_type STRING COMMENT 'Token from Silver failure_reasons (see Code/Silver.py), e.g. null_order_id',
  rejected_record_count BIGINT COMMENT 'Rows in quarantine with this issue (row may count toward multiple issues)',
  ai_business_impact_explanation STRING COMMENT 'Plain-English audit narrative (3–4 sentences)',
  generated_at TIMESTAMP COMMENT 'When this explanation was generated'
) USING DELTA
COMMENT 'UC1 AI DQ audit report — grouped by entity and issue type';
"""
)

DataFrame[]

## 3) Databricks OpenAI-compatible client

In [0]:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
_ws_raw = spark.conf.get("spark.databricks.workspaceUrl")
_ws_url = _ws_raw.replace("https://", "").replace("http://", "").strip("/")

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{_ws_url}/serving-endpoints",
)

## 4) Parse `databricks-gpt-oss-20b` structured JSON (reasoning + text blocks)

In [0]:
def _extract_text_from_blocks(obj) -> str:
    """Pull narrative from structured blocks: list of {type, text} or {type, summary}."""
    if obj is None:
        return ""
    if isinstance(obj, list):
        parts = []
        for block in obj:
            if not isinstance(block, dict):
                if isinstance(block, str) and block.strip():
                    parts.append(block.strip())
                continue
            if block.get("type") == "text":
                t = block.get("text")
                if isinstance(t, str) and t.strip():
                    parts.append(t.strip())
                elif isinstance(t, list):
                    parts.append(" ".join(str(x) for x in t if x is not None))
            # Some endpoints use a single "text" key without type
            elif "text" in block and block.get("text"):
                parts.append(str(block["text"]).strip())
        if parts:
            return "\n\n".join(parts)
        return ""
    if isinstance(obj, dict):
        if obj.get("type") == "text" and "text" in obj:
            return str(obj["text"]).strip()
        if "text" in obj and obj["text"] is not None:
            return str(obj["text"]).strip()
    return ""


def extract_text_from_gpt_oss_content(raw) -> str:
    """Return final narrative text. `message.content` may be str, list[dict], or dict (not always a string)."""
    if raw is None:
        return ""

    # OpenAI / Databricks may return parsed list of blocks directly (no .strip())
    if isinstance(raw, (list, dict)):
        out = _extract_text_from_blocks(raw)
        if out:
            return out
        if isinstance(raw, dict):
            return str(raw.get("content", raw)).strip()
        if isinstance(raw, list):
            flat = [str(x).strip() for x in raw if x is not None and str(x).strip()]
            if flat:
                return "\n\n".join(flat)
        return ""

    s = str(raw).strip() if not isinstance(raw, str) else raw.strip()
    if not s:
        return ""

    # String body: JSON array of typed blocks
    try:
        parsed = json.loads(s)
        out = _extract_text_from_blocks(parsed)
        if out:
            return out
        if isinstance(parsed, dict) and "text" in parsed:
            return str(parsed["text"]).strip()
    except json.JSONDecodeError:
        pass

    # Fallback: regex for embedded JSON text blocks
    for m in re.finditer(
        r'\{\s*"type"\s*:\s*"text"\s*,\s*"text"\s*:\s*"((?:[^"\\]|\\.)*)"\s*\}',
        s,
        re.DOTALL,
    ):
        try:
            return bytes(m.group(1), "utf-8").decode("unicode_escape").replace('\\"', '"')
        except Exception:
            return m.group(1).replace('\\"', '"')

    return s


def call_audit_llm(prompt: str, max_tokens: int = 600) -> str:
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You write clear compliance and audit narratives for finance teams. Be specific: cite the field name, the issue code, and the count. Use plain English.",
            },
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.2,
    )
    raw = ""
    if resp.choices and resp.choices[0].message:
        raw = resp.choices[0].message.content or ""
    return extract_text_from_gpt_oss_content(raw)



## 5) Prompt template

In [0]:
DQ_PROMPT_TEMPLATE = """You are assisting GlobalMart's finance and internal audit team before an external audit.

Data quality finding (grouped — one narrative per issue type, not per row):
- Entity: {entity_name}
- Primary field affected: {field_affected}
- Issue code (from the Silver harmonization pipeline): {issue_type}
- Number of quarantined records with this issue: {rejected_record_count}

Your task: Write exactly 3 to 4 sentences in plain English for a non-technical finance auditor.

The narrative must:
1) Open with the count ({rejected_record_count} records) and name the field and issue code once.
2) Explain what value or pattern caused rejection and why these rows are excluded from the trusted analytics layer.
3) State one specific audit or management report that could be wrong if this issue were ignored.

Do not mention Python, Spark, Databricks, or machine learning. Do not repeat this instruction block."""


def build_prompt(entity_name: str, field_affected: str, issue_type: str, count: int) -> str:
    return DQ_PROMPT_TEMPLATE.format(
        entity_name=entity_name,
        field_affected=field_affected,
        issue_type=issue_type,
        rejected_record_count=count,
    )


## 6) Aggregate quarantine rows by **issue type** (explode `failure_reasons`)

In [0]:
def build_issue_summary(spark_session):
    parts = []
    for full_table, ent in QUARANTINE_SOURCES:
        try:
            df = spark_session.table(full_table)
        except Exception as e:
            print(f"Skip (not found or no access): {full_table} — {e}")
            continue
        if "failure_reasons" not in df.columns:
            print(f"Skip (no failure_reasons column): {full_table}")
            continue
        parts.append(
            df.select(
                F.lit(ent).alias("entity_name"),
                F.explode(F.split(F.col("failure_reasons"), ";")).alias("issue_raw"),
            )
            .withColumn("issue_type", F.trim(F.col("issue_raw")))
            .filter(F.col("issue_type") != "")
            .groupBy("entity_name", "issue_type")
            .agg(F.count("*").alias("rejected_record_count"))
        )
    if not parts:
        schema = StructType(
            [
                StructField("entity_name", StringType(), True),
                StructField("issue_type", StringType(), True),
                StructField("rejected_record_count", LongType(), True),
            ]
        )
        return spark_session.createDataFrame([], schema)
    out = parts[0]
    for p in parts[1:]:
        out = out.unionByName(p)
    return out


summary_df = build_issue_summary(spark)
summary_df.display()


entity_name,issue_type,rejected_record_count
customers,Not_valid_segment_or_null,374
orders,null_ship_mode,231
transactions,null_quantity,238
transactions,null_payment_type,88
returns,null_return_reason_or_not_valid,35
returns,null_refund_amount_or_not_valid,28
returns,invalid_return_status,45
returns,unparseable_return_date,33


## 7) Generate explanations and write **`globalmart.gold.dq_audit_report`**

In [0]:
rows_out = []
generated_at = datetime.now(timezone.utc)
issue_rows = summary_df.collect()

if not issue_rows:
    print("No quarantine issues found — writing empty report (overwrite).")
else:
    for r in issue_rows:
        ent = r["entity_name"]
        issue = r["issue_type"]
        cnt = int(r["rejected_record_count"])
        field = ISSUE_TO_FIELD.get(issue, "(see Silver rule for this entity)")
        prompt = build_prompt(ent, field, issue, cnt)
        explanation = call_audit_llm(prompt)
        rows_out.append(
            (
                ent,
                field,
                issue,
                cnt,
                explanation,
                generated_at,
            )
        )

schema_out = StructType(
    [
        StructField("entity_name", StringType(), False),
        StructField("field_affected", StringType(), True),
        StructField("issue_type", StringType(), False),
        StructField("rejected_record_count", LongType(), False),
        StructField("ai_business_impact_explanation", StringType(), True),
        StructField("generated_at", TimestampType(), False),
    ]
)

report_df = spark.createDataFrame(rows_out, schema_out)
(
    report_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"Written {len(rows_out)} rows to {GOLD_TABLE}")

Written 8 rows to globalmart.gold.dq_audit_report


## 8) Validate — sample audit narrative

In [0]:
display(spark.table(GOLD_TABLE).orderBy("entity_name", "issue_type"))

entity_name,field_affected,issue_type,rejected_record_count,ai_business_impact_explanation,generated_at
customers,segment,Not_valid_segment_or_null,374,"374 records in the customers entity were quarantined because the segment field contained invalid or null values (issue code: Not_valid_segment_or_null). These rows were rejected because the segment values did not match any of the approved categories or were missing entirely, so they could not be reliably used in the trusted analytics layer. As a result, any analysis that relies on customer segmentation—such as the quarterly sales performance report—could present misleading totals or percentages if these records were included.",2026-03-27T10:38:14.300Z
orders,ship_mode,null_ship_mode,231,"231 records in the orders entity were quarantined due to the null_ship_mode issue code on the ship_mode field. These rows had a missing or null value in the ship_mode column, which violates the business rule that every order must specify a shipping method, so they were removed from the trusted analytics layer. If ignored, the monthly sales performance report that aggregates revenue by shipping method could overstate or misclassify shipments.",2026-03-27T10:38:14.300Z
returns,return_status,invalid_return_status,45,"There are 45 records flagged with the issue code **invalid_return_status** in the **return_status** field. These rows contain return_status values that are not part of the approved list (for example, “unknown” or “pending”), so they are excluded from the trusted analytics layer to prevent corrupting downstream calculations. If this issue were ignored, the monthly sales performance report’s return‑rate metric could be inflated, leading to misleading conclusions about product quality.",2026-03-27T10:38:14.300Z
returns,refund_amount,null_refund_amount_or_not_valid,28,"28 records were quarantined because the refund_amount field had a null_refund_amount_or_not_valid issue code. These rows contained refund_amount values that were either null or not a valid numeric amount, violating the rule that requires a positive numeric value for refunds, so they were excluded from the trusted analytics layer. If this issue were ignored, the company’s Return on Sales report could understate the true refund liability, leading to misstated net revenue.",2026-03-27T10:38:14.300Z
returns,return_reason,null_return_reason_or_not_valid,35,"35 records were flagged for the **return_reason** field with issue code **null_return_reason_or_not_valid**. These rows contain either missing (null) or non‑standard values that do not match the approved list of return reasons, so they were quarantined to prevent inaccurate calculations in the trusted analytics layer. If this issue were ignored, the **Returns Analysis Report**—which informs management about the volume and reasons for product returns—could present misleading totals and trends.",2026-03-27T10:38:14.300Z
returns,return_date,unparseable_return_date,33,"33 records were quarantined due to an unparseable_return_date issue in the return_date field. The problem was that the dates were entered in an inconsistent or invalid format, preventing the system from converting them into a standard date value, so these rows cannot be used in the trusted analytics layer. Because the return dates are missing, any calculations that rely on them—such as the period‑to‑period return volume—will be incomplete. If this issue were ignored, the monthly returns reconciliation report could report incorrect totals, misleading management and auditors.",2026-03-27T10:38:14.300Z
transactions,payment_type,null_payment_type,88,"88 records have been quarantined due to a null_payment_type issue in the payment_type field. These rows contain missing or null values in the payment_type column, which prevents accurate categorization of transaction methods and violates the data quality rule that requires a valid payment type for every transaction. Because the trusted analytics layer relies on complete p